# Biosecurity Sentinel — Live Demo

This notebook runs the full assessment pipeline on 5 molecules covering the full risk spectrum:
from harmless (caffeine) to a nerve agent structural analog.

**No trained model weights required for this demo** — the structural alert layer runs deterministically.
The ML prediction outputs are shown but labelled as untrained until GPU compute is available.


In [ ]:
import sys
sys.path.insert(0, '..')

from core.assessment_engine import AssessmentEngine
import json

engine = AssessmentEngine(model_path=None)
print('Engine loaded. Structural alert library active.')

## Molecule 1 — Caffeine (expected: NEGLIGIBLE)

In [ ]:
result = engine.assess('Cn1cnc2c1c(=O)n(C)c(=O)n2C', name='Caffeine')
print(f"Risk Level    : {result['risk_level']}")
print(f"Status        : {result['status']}")
print(f"Alerts        : {len(result['structural_alerts'])} triggered")
print(f"Recommendation: {result['recommendation']}")
print(f"MW            : {result['descriptors'].get('molecular_weight', 'N/A')} g/mol")
print(f"logP          : {result['descriptors'].get('logp', 'N/A')}")

## Molecule 2 — Ibuprofen (expected: LOW)

In [ ]:
result = engine.assess('CC(C)Cc1ccc(cc1)C(C)C(=O)O', name='Ibuprofen')
print(f"Risk Level    : {result['risk_level']}")
print(f"Status        : {result['status']}")
print(f"Alerts        : {len(result['structural_alerts'])} triggered")
print(f"Recommendation: {result['recommendation']}")

## Molecule 3 — Warfarin (expected: MODERATE)

In [ ]:
result = engine.assess('OC=1C(=O)c2ccccc2C=1CC(=O)c1ccccc1', name='Warfarin')
print(f"Risk Level    : {result['risk_level']}")
print(f"Status        : {result['status']}")
print(f"Alerts        : {len(result['structural_alerts'])} triggered")
print(f"Recommendation: {result['recommendation']}")

## Molecule 4 — Ciprofloxacin (expected: LOW)

In [ ]:
result = engine.assess('OC(=O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O', name='Ciprofloxacin')
print(f"Risk Level    : {result['risk_level']}")
print(f"Status        : {result['status']}")
print(f"Alerts        : {len(result['structural_alerts'])} triggered")
print(f"Conformer     : {result['conformer_method']}")
print(f"Recommendation: {result['recommendation']}")

## Molecule 5 — Organophosphate Fluoride Analog (expected: CRITICAL / HALT)

This is a structural analog of nerve agents containing the P(=O)(F) scaffold.
**The structural alert fires before the ML model runs.**

In [ ]:
result = engine.assess('CP(=O)(OCC)F', name='Organophosphate fluoride (test compound)')
print(f"Risk Level    : {result['risk_level']}")
print(f"Status        : {result['status']}")
print()
print('STRUCTURAL ALERTS TRIGGERED:')
for alert in result['structural_alerts']:
    print(f"  [{alert['severity']}] {alert['name']}")
    print(f"           Category : {alert['category']}")
    print(f"           Reference: {alert['reference']}")
print()
print(f"Recommendation: {result['recommendation']}")

## Summary Table

In [ ]:
molecules = [
    ('Caffeine',               'Cn1cnc2c1c(=O)n(C)c(=O)n2C'),
    ('Ibuprofen',              'CC(C)Cc1ccc(cc1)C(C)C(=O)O'),
    ('Ciprofloxacin',          'OC(=O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O'),
    ('Warfarin',               'OC=1C(=O)c2ccccc2C=1CC(=O)c1ccccc1'),
    ('Organophosphate analog', 'CP(=O)(OCC)F'),
]

print(f"{'Molecule':<30} {'Risk Level':<20} {'Alerts':<8} {'Status'}")
print('-' * 75)
for name, smiles in molecules:
    r = engine.assess(smiles, name=name)
    print(f"{name:<30} {r['risk_level']:<20} {len(r['structural_alerts']):<8} {r['status']}")

---

## What happens after GPU training

Once the SchNet encoder is pretrained on QM9 (130k molecules) and fine-tuned on Tox21:

- The `predictions` field will contain real toxicity scores, hepatic clearance values, and dual-use risk scores
- Each prediction will have a 95% confidence interval from 30 Monte Carlo Dropout samples
- Molecules that are structurally unlike anything in training will be flagged `UNKNOWN` rather than silently mispredicted
- The enantiomer test (R vs S stereoisomers of the same molecule) will show the 3D geometry awareness of SchNet vs. 2D fingerprint baselines

The structural alert layer shown in this demo is fully operational and deployed independently of model training status.